In [ ]:
%matplotlib inline

In [ ]:
from __future__ import annotations

from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ExifTags

In [ ]:
def read_image(image_path: str | Path) -> tuple[np.ndarray, np.ndarray]:
    image_path = Path(image_path)

    bgr = cv2.imread(str(image_path))
    if bgr is None:
        raise ValueError(f"Cannot read image: {image_path}")

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)

    return rgb, gray


def has_camera_exif(image_path: str | Path) -> bool:
    image = Image.open(image_path)
    exif = image.getexif()

    if not exif:
        return False

    tag_names = {
        ExifTags.TAGS.get(k, k): v
        for k, v in exif.items()
    }

    camera_tags = {
        "Make",
        "Model",
        "LensModel",
        "DateTimeOriginal",
        "ExposureTime",
        "FNumber",
        "ISOSpeedRatings",
        "FocalLength",
    }

    return any(tag in tag_names for tag in camera_tags)


def normalize_uint8(image: np.ndarray) -> np.ndarray:
    image = image.astype(np.float32)
    image = image - image.min()
    image = image / (image.max() + 1e-8)

    return (image * 255).astype(np.uint8)


def blur_score(gray: np.ndarray) -> float:
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())


def noise_score(gray: np.ndarray) -> float:
    gray_f = gray.astype(np.float32)
    smooth = cv2.GaussianBlur(gray_f, (5, 5), 0)
    noise = gray_f - smooth

    return float(np.std(noise))


def glare_mask(rgb: np.ndarray) -> np.ndarray:
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    _, s, v = cv2.split(hsv)

    bright = v > 245
    low_saturation = s < 40

    mask = bright & low_saturation

    return mask.astype(np.uint8) * 255


def glare_score(rgb: np.ndarray) -> float:
    mask = glare_mask(rgb)

    return float((mask > 0).mean())


def edge_map(gray: np.ndarray) -> np.ndarray:
    return cv2.Canny(gray, 80, 160)


def high_frequency_map(gray: np.ndarray) -> np.ndarray:
    gray_f = gray.astype(np.float32)
    blurred = cv2.GaussianBlur(gray_f, (21, 21), 0)
    high_freq = gray_f - blurred

    return normalize_uint8(np.abs(high_freq))


def moire_score(gray: np.ndarray) -> float:
    gray_f = gray.astype(np.float32)

    blurred = cv2.GaussianBlur(gray_f, (21, 21), 0)
    high_freq = gray_f - blurred

    spectrum = np.fft.fft2(high_freq)
    spectrum = np.fft.fftshift(spectrum)
    magnitude = np.log1p(np.abs(spectrum))

    h, w = magnitude.shape
    cy, cx = h // 2, w // 2

    y, x = np.ogrid[:h, :w]
    radius = min(h, w) // 10

    center_mask = (x - cx) ** 2 + (y - cy) ** 2 <= radius ** 2
    outer_mask = ~center_mask

    high_freq_energy = magnitude[outer_mask].mean()
    total_energy = magnitude.mean() + 1e-8

    return float(high_freq_energy / total_energy)


def straight_lines_score(gray: np.ndarray) -> float:
    edges = edge_map(gray)

    lines = cv2.HoughLinesP(
        edges,
        rho=1,
        theta=np.pi / 180,
        threshold=120,
        minLineLength=min(gray.shape[:2]) // 4,
        maxLineGap=20,
    )

    if lines is None:
        return 0.0

    return float(len(lines))


def sigmoid(x: float) -> float:
    return float(1 / (1 + np.exp(-x)))

In [ ]:
def probability_photo_of_monitor(
    *,
    has_exif: bool,
    blur: float,
    noise: float,
    glare: float,
    moire: float,
    lines: float,
) -> float:
    score = 0.0

    # EXIF камеры — сильный сигнал
    if has_exif:
        score += 2.5

    # Мутность
    if blur < 80:
        score += 1.3
    elif blur < 150:
        score += 0.9
    elif blur < 300:
        score += 0.4
    else:
        score -= 0.6

    # Шум камеры
    if noise > 12:
        score += 1.1
    elif noise > 8:
        score += 0.7
    elif noise > 5:
        score += 0.3
    else:
        score -= 0.4

    # Блики
    if glare > 0.03:
        score += 1.0
    elif glare > 0.01:
        score += 0.5

    # Муар / сетка
    if moire > 1.12:
        score += 1.0
    elif moire > 1.08:
        score += 0.5

    # Длинные линии
    if lines > 40:
        score += 0.6
    elif lines > 20:
        score += 0.3

    score -= 1.2

    return sigmoid(score) * 100

In [ ]:
def analyze_image(image_path: str | Path) -> dict:
    image_path = Path(image_path)

    rgb, gray = read_image(image_path)

    exif = has_camera_exif(image_path)
    blur = blur_score(gray)
    noise = noise_score(gray)
    glare = glare_score(rgb)
    moire = moire_score(gray)
    lines = straight_lines_score(gray)

    probability = probability_photo_of_monitor(
        has_exif=exif,
        blur=blur,
        noise=noise,
        glare=glare,
        moire=moire,
        lines=lines,
    )

    if probability >= 70:
        label = "photo_of_monitor"
    elif probability <= 30:
        label = "not_photo_of_monitor"
    else:
        label = "uncertain"

    return {
        "image_path": str(image_path),
        "label": label,
        "probability": probability,
        "has_camera_exif": exif,
        "blur_score": blur,
        "noise_score": noise,
        "glare_score": glare,
        "moire_score": moire,
        "straight_lines_score": lines,
        "rgb": rgb,
        "gray": gray,
        "edges": edge_map(gray),
        "glare_mask": glare_mask(rgb),
        "high_frequency_map": high_frequency_map(gray),
    }

In [ ]:
def show_result(result: dict) -> None:
    probability = result["probability"]
    label = result["label"]

    title = (
        f"{label} | probability: {probability:.1f}%\n"
        f"EXIF: {result['has_camera_exif']} | "
        f"blur: {result['blur_score']:.1f} | "
        f"noise: {result['noise_score']:.1f} | "
        f"glare: {result['glare_score']:.4f} | "
        f"moire: {result['moire_score']:.3f} | "
        f"lines: {result['straight_lines_score']:.0f}"
    )

    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    fig.suptitle(title, fontsize=13)

    axes[0, 0].imshow(result["rgb"])
    axes[0, 0].set_title("Original")

    axes[0, 1].imshow(result["gray"], cmap="gray")
    axes[0, 1].set_title("Grayscale")

    axes[0, 2].imshow(result["edges"], cmap="gray")
    axes[0, 2].set_title("Edges")

    axes[1, 0].imshow(result["glare_mask"], cmap="gray")
    axes[1, 0].set_title("Glare mask")

    axes[1, 1].imshow(result["high_frequency_map"], cmap="gray")
    axes[1, 1].set_title("High-frequency map")

    prob_bar = np.zeros((80, 400), dtype=np.uint8)
    filled = int(probability / 100 * prob_bar.shape[1])
    prob_bar[:, :filled] = 255

    axes[1, 2].imshow(prob_bar, cmap="gray", vmin=0, vmax=255)
    axes[1, 2].set_title(f"Photo monitor probability: {probability:.1f}%")

    for ax in axes.ravel():
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
image_path = "path/to/image.jpg"

result = analyze_image(image_path)

print(f"Файл: {result['image_path']}")
print(f"Класс: {result['label']}")
print(f"Вероятность фото монитора: {result['probability']:.2f}%")
print(f"EXIF камеры: {result['has_camera_exif']}")
print(f"Blur score: {result['blur_score']:.2f}")
print(f"Noise score: {result['noise_score']:.2f}")
print(f"Glare score: {result['glare_score']:.6f}")
print(f"Moire score: {result['moire_score']:.4f}")
print(f"Straight lines score: {result['straight_lines_score']:.0f}")

show_result(result)

In [ ]:
from pathlib import Path

image_dir = Path("path/to/images")

image_paths = []
for pattern in ["*.jpg", "*.jpeg", "*.png", "*.webp"]:
    image_paths.extend(image_dir.glob(pattern))

results = []

for image_path in image_paths:
    try:
        result = analyze_image(image_path)

        results.append({
            "path": str(image_path),
            "label": result["label"],
            "probability": result["probability"],
            "has_camera_exif": result["has_camera_exif"],
            "blur_score": result["blur_score"],
            "noise_score": result["noise_score"],
            "glare_score": result["glare_score"],
            "moire_score": result["moire_score"],
            "straight_lines_score": result["straight_lines_score"],
        })

        print(
            f"{image_path.name}: "
            f"{result['label']} | "
            f"{result['probability']:.2f}%"
        )

    except Exception as e:
        print(f"ERROR {image_path}: {e}")

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df = df.sort_values("probability", ascending=False)

df

In [ ]:
top_n = 5

for _, row in df.head(top_n).iterrows():
    result = analyze_image(row["path"])
    show_result(result)